# Data Preprocessing

In [3]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

input_file = processed_path / "cleaned_cicids2017.csv"

df = pd.read_csv(input_file)

print("Cleaned dataset loaded successfully!")
print("Dataset shape:", df.shape)

Cleaned dataset loaded successfully!
Dataset shape: (2520798, 79)


In [4]:
print(df.head())

   Destination Port  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0             54865              3                  2                       0   
1             55054            109                  1                       1   
2             55055             52                  1                       1   
3             46236             34                  1                       1   
4             54863              3                  2                       0   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                           12                            0   
1                            6                            6   
2                            6                            6   
3                            6                            6   
4                           12                            0   

   Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet Length Mean  \
0                      6                      6            

In [5]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 2520798 entries, 0 to 2520797
Data columns (total 79 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  Flow IAT Mi

In [6]:
print(df["Label"].value_counts())

Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [7]:
X = df.drop("Label", axis=1)
y = df["Label"]

In [8]:
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (2520798, 78)
Target shape: (2520798,)


In [10]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

In [11]:
label_mapping = dict(
    zip(label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_))
)

print(label_mapping)

{'BENIGN': np.int64(0), 'Bot': np.int64(1), 'DDoS': np.int64(2), 'DoS GoldenEye': np.int64(3), 'DoS Hulk': np.int64(4), 'DoS Slowhttptest': np.int64(5), 'DoS slowloris': np.int64(6), 'FTP-Patator': np.int64(7), 'Heartbleed': np.int64(8), 'Infiltration': np.int64(9), 'PortScan': np.int64(10), 'SSH-Patator': np.int64(11), 'Web Attack � Brute Force': np.int64(12), 'Web Attack � Sql Injection': np.int64(13), 'Web Attack � XSS': np.int64(14)}


In [12]:
print(X.dtypes.value_counts())

int64      54
float64    24
Name: count, dtype: int64


In [13]:
non_numeric_columns = X.select_dtypes(exclude=["number"]).columns

print("Non-numeric columns:")
print(non_numeric_columns)

Non-numeric columns:
Index([], dtype='str')


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [15]:
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training features: (2016638, 78)
Testing features: (504160, 78)
Training labels: (2016638,)
Testing labels: (504160,)


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [17]:
import joblib

joblib.dump(
    scaler,
    "../models/standard_scaler.pkl"
)

print("Scaler saved successfully.")

Scaler saved successfully.


In [18]:
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (2016638, 78)
X_test_scaled shape: (504160, 78)


In [19]:
print("Missing values in training data:",
      pd.isna(X_train_scaled).sum())

print("Missing values in testing data:",
      pd.isna(X_test_scaled).sum())

Missing values in training data: 0
Missing values in testing data: 0


In [20]:
from pathlib import Path

processed_path = Path("../data/processed")

In [21]:
X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X.columns
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X.columns
)

X_train_scaled_df.to_csv(
    processed_path / "X_train.csv",
    index=False
)

X_test_scaled_df.to_csv(
    processed_path / "X_test.csv",
    index=False
)

In [ ]:
pd.DataFrame(y_train, columns=["Label"]).to_csv(
    processed_path / "y_train.csv",
    index=False
)

pd.DataFrame(y_test, columns=["Label"]).to_csv(
    processed_path / "y_test.csv",
    index=False
)

In [ ]:
label_mapping_df = pd.DataFrame({
    "Original_Label": label_encoder.classes_,
    "Encoded_Label": label_encoder.transform(label_encoder.classes_)
})

label_mapping_df.to_csv(
    processed_path / "label_mapping.csv",
    index=False
)